In [ ]:
import time
import random
import re
import hashlib
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import spacy

In [ ]:
nlp = spacy.load("de_core_news_md")
BASE_SEARCH = "https://www.zeit.de/suche/index"
CLIMATE_KEYWORDS = ["klima", "klimawandel", "klimakrise", "erderwärmung", "globale erwärmung", "treibhauseffekt", "treibhausgas",
    "co2", "kohlendioxid", "emission", "emissionen", "klimaschutz", "klimapolitik", "klimaneutral", "dekarbonisierung",
    
    "klimaziel", "klimaziele", "co2-preis", "emissionshandel", "klimaschutzgesetz", "paris-abkommen", "pariser abkommen",
    "eu-klimapaket", "green deal", "klimaplan", "klimabericht", "ipcc",

    "energiewende", "erneuerbare energien", "windkraft", "solarenergie", "photovoltaik", "wasserstoff",
    "kohlekraft", "kohleausstieg", "atomkraft", "stromnetz", "netzausbau", "e-mobilität", "verbrennerverbot", "industrieemissionen",
    
    "hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter",
    "gletscherschmelze", "meeresspiegel", "wasserknappheit", "klimafolgen", "artensterben",

    "klimarisiko", "nachhaltigkeit", "nachhaltige investitionen", "esg", "grüne finanzierung", "klimafonds",
    "energiepreise", "strompreis", "gaspreis",

    "landwirtschaft", "trockenheit", "ernteschäden", "umweltschutz", "biodiversität", "naturschutz", "ökosystem", "umweltpolitik",

    "fridays for future", "klimaaktivisten", "klimaprotest", "letzte generation", "klimabewegung", "klimadebatte"]

STRICT_KEYWORDS = ["klima", "klimakrise", "klimawandel", "erderwärmung", "co2", "kohlendioxid", "emission", "emissionen",
    "energiewende", "klimaschutz", "klimaneutral", "treibhauseffekt", "treibhausgas",
    "hitzewelle", "dürre", "hochwasser", "wasserknappheit", "starkregen", "waldbrand"]
YEARS = range(2020,2026)
PAGES = range(1,40)
OUTPUT_FILE = "scrapped_zeit.csv"
INVALID_URL_PATTERNS = ["/video/","/angebote/","/spiele/","/campus/","/index"]
CLASSIFICATION_KEYWORDS = {
    "Climate_Policy": [
        "gesetz", "politik", "regierung", "beschluss", "verordnung", "ziel", "klimaziel", "bundestag", "eu", "parlament", "ministerium"],
    "Climate_Science": ["studie", "forschung", "wissenschaft", "ipcc", "daten", "analyse", "bericht", "modell", "forscher"],
    "Energy_Transition": ["energiewende", "erneuerbar", "solar", "windkraft", "kohlekraft", "atomkraft", "wasserstoff", "stromnetz"],
    "Climate_Economy": ["kosten", "industrie", "wirtschaft", "markt", "unternehmen", "investition", "preis", "arbeitsplätze"],
    "Climate_Activism": ["protest", "demonstration", "aktivisten", "fridays for future", "ngo", "bewegung"],
    "Climate_Impact": ["hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter"],
    "Climate_Geopolitics": ["china", "usa", "eu", "russland", "international", "global", "weltweit", "g7", "g20"],
    "Climate_Opinion": ["meinung", "kommentar", "kolumne", "leitartikel", "debatte", "gastbeitrag"]}

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("user-agent=Mozilla/5.0")
chrome_options.add_argument(r"--user-data-dir=C:/selenium_chrome_profile")
driver = webdriver.Chrome(
service=Service(ChromeDriverManager().install()),
options=chrome_options)
wait = WebDriverWait(driver,20)

In [ ]:
def human_sleep(a=2,b=4):
    time.sleep(random.uniform(a,b))
def safe_get(url):
    try:
        driver.get(url)
        human_sleep()
        return True
    except:
        return False
def extract_actors(text):
    doc = nlp(text[:4000])
    return sorted(set(ent.text for ent in doc.ents if ent.label_ in ("PER","ORG")))
def sentence_count(text):
    return sum(1 for _ in nlp(text).sents)
def extract_date():
    try:
        return driver.find_element(By.TAG_NAME,"time").get_attribute("datetime")
    except:
        return None
def extract_author():
    try:
        authors = driver.find_elements(
            By.CSS_SELECTOR,'a[rel="author"] span[itemprop="name"]')
        return ", ".join(a.text.strip() for a in authors)
    except:
        return None
def extract_article():
    paragraphs = driver.find_elements(By.CSS_SELECTOR,"article p")
    clean = []
    for p in paragraphs:
        text = p.text.strip()
        if len(text) < 40:
            continue
        if any(x in text.lower() for x in
               ["anzeige","newsletter","abonnieren"]):
            continue
        clean.append(text)
    if len(clean) < 3:
        return None,None
    intro = clean[0]
    content = re.sub(r"\s+"," "," ".join(clean))
    return intro,content
def strict_relevance(title,content):
    title = title.lower()
    content_l = content.lower()
    lead = " ".join(content_l.split()[:300])
    title_hit = any(k in title for k in STRICT_KEYWORDS)
    lead_hits = sum(k in lead for k in STRICT_KEYWORDS)
    return title_hit or lead_hits >= 3
def classify_article(title,intro,content):
    text = f"{title} {intro} {' '.join(content.split()[:300])}".lower()
    scores = {}
    for cat,keywords in CLASSIFICATION_KEYWORDS.items():
        scores[cat] = sum(k in text for k in keywords)
    best = max(scores,key=scores.get)
    return best if scores[best] > 0 else "Other_Climate"

In [ ]:
rows = []
visited_urls = set()
visited_hashes = set()

In [ ]:
for keyword in CLIMATE_KEYWORDS:
    for year in YEARS:
        query = f"{keyword} {year}"
        for page in PAGES:
            search_url = f"{BASE_SEARCH}?p={page}&q={query}"
            print(query,"| Page",page)
            if not safe_get(search_url):
                continue
            links = driver.find_elements(By.CSS_SELECTOR,"article a[href]")
            urls = []
            for a in links:
                try:
                    href = a.get_attribute("href")
                    if not href:
                        continue
                    if not href.startswith("https://www.zeit.de/"):
                        continue
                    if any(bad in href for bad in INVALID_URL_PATTERNS):
                        continue
                    if href in visited_urls:
                        continue
                    visited_urls.add(href)
                    urls.append(href)
                except:
                    continue
            for url in urls:
                if not safe_get(url):
                    continue
                try:
                    wait.until(
                    EC.presence_of_element_located((By.TAG_NAME,"h1")))
                    headline = driver.find_element(By.TAG_NAME,"h1").text.strip()
                    intro,content = extract_article()
                    if not content:
                        continue
                    if not strict_relevance(headline,content):
                        continue
                    content_hash = hashlib.md5(
                        content[:1000].encode("utf-8")).hexdigest()
                    if content_hash in visited_hashes:
                        continue
                    visited_hashes.add(content_hash)
                    actors = extract_actors(content)
                    rows.append({

                        "URL": url,
                        "Source": "DIE ZEIT",
                        "Language": "de",
                        "Published_Date": extract_date(),
                        "Keyword_Matched": query,
                        "Article_Classification": classify_article(headline,intro,content),
                        "Headline": headline,
                        "Intro": intro,
                        "Content": content,
                        "Content_Length": len(content),
                        "Sentence_Count": sentence_count(content),
                        "Actors": ", ".join(actors),
                        "Actor_Count": len(actors),
                        "Author": extract_author()
                    })
                    print("Saved:")
                    human_sleep()
                except:
                    continue

In [ ]:
df = pd.DataFrame(rows)
df.drop_duplicates(subset=["URL"],inplace=True)
df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")
driver.quit()
print("Saved File:",OUTPUT_FILE)
print("Total number of articles scrapped in Zeit:",len(df))